In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")
    wait.until(EC.presence_of_element_located((By.ID, "username")))

    # Clearly invalid data that still passes client-side length rules,
    # so it reaches the server, which must reject it (verified: formError
    # renders in div.form-alert in login.jsx; nothing is written on failure).
    driver.find_element(By.ID, "username").send_keys("not-an-email-xyz")
    driver.find_element(By.ID, "password").send_keys("wrongpass123")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    alert = wait.until(EC.visibility_of_element_located((By.XPATH, "//div[contains(@class, 'form-alert')]")))
    assert alert.text.strip(), "Error alert appeared but is empty."
    print("Rejection message:", alert.text.strip())

    # Invalid data must not be accepted: still anonymous, no session
    assert driver.find_elements(By.ID, "username"), "Login form disappeared."
    assert not driver.find_elements(By.XPATH, "//nav[@aria-label='Staff']"), \
        "Authenticated UI shown for invalid credentials."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert not token, "Session was created from invalid data."
    print("PASS: Invalid data submission rejected")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("53_invalid_data_FAIL.png")
finally:
    driver.quit()